## Pinecone Vector Database (official `pinecone` client)

This notebook tests creating a **Pinecone** vector store from a PDF document, using the official Pinecone SDK directly instead of the `langchain-pinecone` integration. Steps:
1. Load the PDF from the `00_data` folder using `PyPDFLoader`.
2. Split it into chunks with a text splitter.
3. Embed the chunks using HuggingFace sentence-transformer embeddings.
4. Create (or connect to) a Pinecone index, upsert the embedding vectors, then run a similarity search query against it.

**Prerequisites:**
- `pip install pinecone`
- A Pinecone account with an API key, set as `PINECONE_API_KEY` in the `.env` file at the project root.

> Note: `langchain-pinecone` is not used here - it does not support Python 3.14 yet. LangChain still handles loading, splitting and embedding; only the vector store calls are made with the Pinecone client.

In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("00_data/attention.pdf")
docs = loader.load()
print(f"Number of pages loaded: {len(docs)}")
docs[0]

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_15634/3750619610.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/bk/brew-global-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages loaded: 15


Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '00_data/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nl

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"Number of chunks: {len(split_docs)}")

Number of chunks: 52


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5330.34it/s]


In [5]:
import time

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
cloud = os.getenv("PINECONE_CLOUD")
region = os.getenv("PINECONE_REGION")
index_name = "pinecone-tutorial-index"

# Create the index if it doesn't already exist
# all-MiniLM-L6-v2 produces 384-dimensional embeddings
if index_name not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud=cloud, region=region),
    )

# A freshly created index takes a few seconds to become queryable
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

index = pc.Index(index_name)

In [6]:
import uuid

# Embed every chunk, then upsert the vectors with the Pinecone client
vectors = embeddings.embed_documents([doc.page_content for doc in split_docs])

records = []
for doc, vector in zip(split_docs, vectors):
    # The chunk text is kept in metadata so the query results can show it back
    # Pinecone rejects null metadata values, so only keep fields that are set
    metadata = {"text": doc.page_content}
    metadata.update(
        {k: v for k, v in doc.metadata.items() if k in ("page", "source") and v is not None}
    )
    records.append({"id": str(uuid.uuid4()), "values": vector, "metadata": metadata})

# Pinecone limits the size of a single upsert request, so send them in batches
batch_size = 100
for i in range(0, len(records), batch_size):
    index.upsert(vectors=records[i : i + batch_size])

print("Upserted", len(records), "vectors into", index_name)
index.describe_index_stats()

Upserted 52 vectors into pinecone-tutorial-index


DescribeIndexStatsResponse(dimension=384, total_vector_count=52, metric='cosine', namespaces=1)

In [7]:
query = "What is self-attention?"
query_vector = embeddings.embed_query(query)

response = index.query(vector=query_vector, top_k=3, include_metadata=True)

for i, match in enumerate(response["matches"]):
    print(f"--- Result {i+1} (page {match['metadata'].get('page')}, score {match['score']:.4f}) ---")
    print(match["metadata"]["text"][:300])
    print()

--- Result 1 (page 2, score 0.5063) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Result 2 (page 5, score 0.4649) ---
PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encounter

--- Result 3 (page 1, score 0.4619) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
t



## Using the Index as a Retriever

The query above calls the Pinecone API directly. A **retriever** wraps that call in LangChain's standard `Runnable` interface, so it can be dropped into a chain or a RAG pipeline exactly like a Chroma or FAISS retriever.

The other notebooks get this for free from `vector_db.as_retriever(...)`, which offers three search types: `similarity`, `similarity_score_threshold` and `mmr`. Here there is no LangChain vector store to call it on, so the adapter is written by hand: subclass `BaseRetriever`, implement `_get_relevant_documents`, and turn each Pinecone match back into a `Document`. That is roughly what `langchain-pinecone` does internally.

Of the three search types, two carry over:

- **`similarity`** - Pinecone's `top_k` is exactly this.
- **`similarity_score_threshold`** - Pinecone returns a score with every match, so the retriever can filter on it. Note the scale differs from the other notebooks: this index uses the `cosine` metric, so scores are raw cosine similarity in `[-1, 1]`, not LangChain's converted 0-1 relevance score.
- **`mmr`** is *not* available. Maximal Marginal Relevance is implemented by the LangChain vector store wrapper re-ranking a candidate pool client-side, not by the Pinecone API. To get it you would fetch a larger `top_k` and run `langchain_community.vectorstores.utils.maximal_marginal_relevance` over the returned vectors yourself.

In [ ]:
from typing import Any, Optional

from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.retrievers import BaseRetriever


class PineconeRetriever(BaseRetriever):
    """Minimal retriever backed by a Pinecone index and a LangChain embedding model."""

    index: Any
    embeddings: Embeddings
    k: int = 3
    # Optional cosine-similarity cut-off; matches scoring below it are dropped
    score_threshold: Optional[float] = None
    # Optional Pinecone metadata filter, applied server-side before ranking
    filter: Optional[dict] = None
    # Optional namespace to search within
    namespace: Optional[str] = None

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> list[Document]:
        response = self.index.query(
            vector=self.embeddings.embed_query(query),
            top_k=self.k,
            include_metadata=True,
            filter=self.filter,
            namespace=self.namespace,
        )

        documents = []
        for match in response["matches"]:
            if self.score_threshold is not None and match["score"] < self.score_threshold:
                continue
            metadata = dict(match["metadata"])
            # The chunk text was stored in metadata at upsert time
            text = metadata.pop("text", "")
            metadata["score"] = match["score"]
            documents.append(Document(page_content=text, metadata=metadata))
        return documents

### 1. Similarity (the default)

Returns the `k` matches whose vectors sit closest to the query vector under the index's `cosine` metric.

In [9]:
retriever = PineconeRetriever(index=index, embeddings=embeddings, k=3)

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")
for i, doc in enumerate(retrieved_docs):
    print(
        f"--- Document {i+1} (page {doc.metadata.get('page')}, "
        f"score {doc.metadata['score']:.4f}) ---"
    )
    print(doc.page_content[:300])
    print()

Retrieved 3 documents
--- Document 1 (page 2, score 0.5063) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Document 2 (page 5, score 0.4649) ---
PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encounter

--- Document 3 (page 1, score 0.4619) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
t



### 2. Score threshold

Pinecone still returns `top_k` matches, so the cut-off is applied on the retriever side: matches scoring below `score_threshold` are dropped and never become `Document`s. A weak query can therefore return fewer than `k` chunks - or none at all.

Cosine similarity for this model runs roughly from `0.5` for a good match down to negative values for an unrelated question, so `0.4` is a reasonable starting cut-off. Print the scores on your own queries before settling on one.

In [10]:
# Compare an on-topic query against an unrelated one
off_topic_query = "How do I bake sourdough bread?"

threshold_retriever = PineconeRetriever(
    index=index, embeddings=embeddings, k=3, score_threshold=0.4
)

for q in (query, off_topic_query):
    docs = threshold_retriever.invoke(q)
    print(f"{q!r} -> {len(docs)} document(s) above the threshold")
    for doc in docs:
        print(f"   score {doc.metadata['score']:.4f}: {doc.page_content[:120]}...")
    print()

'What is self-attention?' -> 3 document(s) above the threshold
   score 0.5063: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where...
   score 0.4649: PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produ...
   score 0.4619: in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to...

'How do I bake sourdough bread?' -> 3 document(s) above the threshold
   score 0.1247: Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metri...
   score 0.1153: Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and ...
   score 0.1044: FFN(x) = max(0,xW 1 +b1)W2 +b2 (2)
While the linear transformations are the same across different positions, they use di...



### 3. Metadata filtering

Unlike FAISS, Pinecone evaluates the filter **server-side, before ranking**, so `top_k` still means "k results" rather than "k candidates that then get thinned out". This is the main practical advantage of a hosted vector database over an in-process index.

Filters use a MongoDB-style syntax over the metadata upserted with each vector: `$eq $ne $gt $gte $lt $lte $in $nin $and $or $exists`. Only the fields actually stored are filterable - the upsert cell above kept `text`, `page` and `source`, so `page` is the useful one here.

Keep metadata small. Pinecone caps it at 40 KB per vector, and the chunk text is already sitting in there.

In [11]:
query_vector = embeddings.embed_query(query)

# Server-side filter: only vectors from pages 5 and up are ranked at all
filtered = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True,
    filter={"page": {"$gte": 5}},
)
print("pages >= 5:", [(m["metadata"]["page"], round(m["score"], 3)) for m in filtered["matches"]])

# $in for a set of values
subset = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True,
    filter={"page": {"$in": [1, 2, 3]}},
)
print("pages 1-3 :", [(m["metadata"]["page"], round(m["score"], 3)) for m in subset["matches"]])

# The retriever takes the same filter
filtered_retriever = PineconeRetriever(
    index=index, embeddings=embeddings, k=3, filter={"page": {"$gte": 5}}
)
print("retriever :", [d.metadata["page"] for d in filtered_retriever.invoke(query)])

pages >= 5: [(5, 0.465), (12, 0.454), (5, 0.446)]
pages 1-3 : [(2, 0.506), (1, 0.462), (2, 0.419)]
retriever : [2, 5, 1]


### 4. Namespaces (partitioning, no LangChain equivalent)

A namespace is a named partition inside one index. Queries touch exactly one namespace, so this is how Pinecone does multi-tenancy: one namespace per customer, per document, or per environment, all sharing one index and one billing unit. There is no equivalent in FAISS or Chroma - you would run separate indexes or separate collections.

A namespace is created implicitly by upserting into it, and the default namespace is `""`.

In [12]:
# Upsert a couple of vectors into a separate namespace
demo_records = [
    {"id": f"demo-{i}", "values": v, "metadata": {"text": t}}
    for i, (t, v) in enumerate(
        zip(
            ["Attention weights are computed with a softmax.", "Sourdough needs a starter."],
            embeddings.embed_documents(
                ["Attention weights are computed with a softmax.", "Sourdough needs a starter."]
            ),
        )
    )
]
index.upsert(vectors=demo_records, namespace="demo")

print("namespaces:", list(index.describe_index_stats()["namespaces"].keys()))

# The default namespace ("") does not see the demo vectors, and vice versa
for ns in ("", "demo"):
    res = index.query(vector=query_vector, top_k=2, include_metadata=True, namespace=ns)
    print(f"namespace {ns or '(default)'!r:11} ->", [round(m["score"], 3) for m in res["matches"]])

namespaces: ['demo', '__default__']
namespace '(default)' -> [0.506, 0.465]
namespace 'demo'      -> [0.406, 0.065]


### 5. Hybrid search (dense + sparse)

Dense vectors miss exact terms - product codes, error numbers, rare acronyms. Hybrid search combines them with a sparse, keyword-style signal. There are two ways to get it here, and they are genuinely different:

**a) Native sparse-dense vectors.** Pinecone can store a sparse vector alongside the dense one and blend both in a single server-side query. It is the better option - one round trip, one ranked list, scoring done where the data lives - but it has real setup costs:

- the index must be created with `metric="dotproduct"`. Cosine and euclidean indexes reject sparse vectors, and the metric cannot be changed after creation, so the index built at the top of this notebook will not do.
- you need a sparse encoder, e.g. `pip install pinecone-text` for BM25.
- the encoder must be *fitted* on your corpus and persisted, since the term statistics have to match at upsert and query time.

```python
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().fit([doc.page_content for doc in split_docs])

# at upsert time, alongside "values"
record["sparse_values"] = bm25_encoder.encode_documents(doc.page_content)

# at query time
index.query(
    vector=dense_vector,
    sparse_vector=bm25_encoder.encode_queries(query),
    top_k=3,
    include_metadata=True,
)
```

To control the dense/sparse balance you scale the two vectors yourself before querying - Pinecone has no `alpha` parameter.

**b) Client-side fusion**, exactly as in the Chroma and FAISS notebooks: a local `BM25Retriever` over `split_docs` plus this notebook's `PineconeRetriever`, merged by `EnsembleRetriever` with Reciprocal Rank Fusion. It needs no change to the index, which is why the cell below uses it - but the BM25 half runs in this process over the in-memory chunks, so it does not scale past what you can hold locally.

In [13]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# BM25 runs locally over the chunks - no Pinecone involved on this half
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 3

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.4, 0.6],  # must sum to 1.0
)

print("keyword only:", [d.metadata.get("page") for d in bm25_retriever.invoke(query)])
print("dense only  :", [d.metadata.get("page") for d in retriever.invoke(query)])
print("hybrid      :", [d.metadata.get("page") for d in hybrid_retriever.invoke(query)])

keyword only: [5, 2, 14]
dense only  : [2, 5, 1]
hybrid      : [2, 5, 1, 5, 2, 14]


### 6. Reranking

Retrieval embeds the query and the chunk *separately*, which is what makes it fast enough to index everything in advance. A **cross-encoder** reads the query and one chunk together and scores the pair - much more accurate, much too slow for the whole corpus. So you retrieve wide (`top_k=20`) and rerank down to the best few.

Pinecone hosts reranking models, so unlike Chroma and FAISS you do not have to download one:

```python
result = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": m["id"], "text": m["metadata"]["text"]} for m in wide["matches"]],
    rank_fields=["text"],
    top_n=3,
    return_documents=True,
)
```

This is a billed API call on your Pinecone account, which is why the cell below uses the local cross-encoder from the other notebooks instead. Swap in the snippet above if you would rather let Pinecone do it.

In [14]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Downloads a small reranking model (~90 MB) on first run
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

wide_retriever = PineconeRetriever(index=index, embeddings=embeddings, k=20)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),
    base_retriever=wide_retriever,
)

print("before rerank:", [d.metadata.get("page") for d in wide_retriever.invoke(query)][:6], "...")
print("after rerank :", [d.metadata.get("page") for d in rerank_retriever.invoke(query)])
print()

for i, doc in enumerate(rerank_retriever.invoke(query)):
    print(f"--- Reranked {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6333.71it/s]


before rerank: [2, 5, 1, 12, 5, 4] ...
after rerank : [1, 4, 4]

--- Reranked 1 (page 1) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is


--- Reranked 2 (page 4) ---
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
i

--- Reranked 3 (page 4) ---
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from t



### Where each technique fits

| Technique | Layer | Pinecone specifics |
|---|---|---|
| Similarity (`top_k`) | search type | Native. |
| Score threshold | search type | Filtered client-side; scores are raw cosine in `[-1, 1]`. |
| MMR | search type | **Not available** - it is client-side re-ranking done by the LangChain wrapper. |
| Metadata filter | query option | Native and **server-side**, so `top_k` is honoured. Best-in-class here. |
| Distance metric | index setting | `cosine`, `euclidean` or `dotproduct`, fixed at `create_index`. `dotproduct` is required for sparse-dense. |
| Namespaces | index setting | Native partitioning; no FAISS or Chroma equivalent. |
| Keyword / BM25 | separate retriever | Either native sparse vectors or a local `BM25Retriever`. |
| Hybrid | composition | Native sparse-dense in one query, or client-side RRF. |
| Reranking | post-processing | Hosted via `pc.inference.rerank`, or a local cross-encoder. |

Compared with the other two notebooks: Pinecone wins on filtering and partitioning because it is a real server, loses MMR because that only exists in the LangChain wrapper, and is the only one of the three that can do hybrid search inside the index itself.